# HalluGuard × MiniCheck integration smoke test

This notebook validates `lrsbrgrn/HalluGuard-Qwen3-4B` as a native model option in the pinned `Liyan06/MiniCheck` framework. All model inference goes through `MiniCheck.score()`; MiniCheck retains document chunking, claim sentence splitting, SentenceFusion, score aggregation, and its fixed `> 0.5` threshold.

Runtime target: Kaggle **GPU T4 x2**. The notebook clones and patches MiniCheck, reads `HF_TOKEN` only from the environment/Kaggle Secrets if available, and saves diagnostics under `/kaggle/working/halluguard_minicheck_smoke`.


## 1. Clean-session setup

Internet access is required for GitHub, PyPI, Hugging Face model, and dataset downloads. The source commit is pinned and patch anchors fail loudly if upstream code differs.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# vLLM 0.10.2 V1 can enter Torch Inductor CUDAGraph paths even with
# enforce_eager=True; a long T4 run reproduced an allocator-state crash.
# V0 plus eager execution avoids that subsystem. Attention stays auto-selected.
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORKING = Path("/kaggle/working")
if not WORKING.exists():
    WORKING = Path.cwd()
SOURCE_DIR = WORKING / "minicheck_halluguard_source"
MINICHECK_COMMIT = "b58b9fa69acbd1015ec970fa65dd752413a053d2"

def run_checked(args, cwd=None):
    print("+", " ".join(map(str, args)))
    return subprocess.run(list(map(str, args)), cwd=cwd, check=True, text=True)

gpu_before = subprocess.run(
    ["nvidia-smi"], text=True, capture_output=True, check=True
).stdout
print(gpu_before)
assert "T4" in gpu_before, "This notebook targets Kaggle T4 GPUs."

if SOURCE_DIR.exists():
    shutil.rmtree(SOURCE_DIR)
run_checked([
    "git", "clone", "--filter=blob:none", "https://github.com/Liyan06/MiniCheck.git", str(SOURCE_DIR)
])
run_checked(["git", "checkout", MINICHECK_COMMIT], cwd=SOURCE_DIR)

run_checked([
    sys.executable, "-m", "pip", "install", "-q",
    "vllm==0.10.2", "transformers==4.57.6", "datasets", "nltk", "scikit-learn",
    "pandas", "huggingface_hub", "accelerate"
])

# Read-only credential handling. Never place token values in notebook source or output.
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN is available from the environment/Kaggle Secrets (value not printed).")
else:
    print("HF_TOKEN is not set; continuing because the referenced public assets may not require it.")


In [ ]:
import ast
from pathlib import Path

mini_path = SOURCE_DIR / "minicheck" / "minicheck.py"
inf_path = SOURCE_DIR / "minicheck" / "inference.py"
mini = mini_path.read_text(encoding="utf-8")
inf = inf_path.read_text(encoding="utf-8")

def replace_exact(text, old, new, label, expected=1):
    found = text.count(old)
    if found != expected:
        raise RuntimeError(f"{label}: expected {expected} anchor(s), found {found}")
    return text.replace(old, new, expected)

allowed_tail = "'Granite-Guardian-3.3-8B']"
mini = replace_exact(
    mini,
    allowed_tail,
    "'Granite-Guardian-3.3-8B', 'HalluGuard-Qwen3-4B']",
    "MiniCheck allowed-model list",
    expected=2,
)

anchor = """        elif model_name == 'Granite-Guardian-3.3-8B':
            if not max_tokens or max_tokens<2048:
                print(\"For Granite Guardian 3.3 - fixing the max_tokens to be 2048\")
                max_tokens=2048

            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )"""
halluguard_constructor = """        elif model_name == 'HalluGuard-Qwen3-4B':
            if not max_tokens or max_tokens < 2:
                raise ValueError(\"HalluGuard max_tokens must be at least 2.\")
            self.model = LLMCheck(
                model_id=model_name,
                tensor_parallel_size=tensor_parallel_size,
                max_tokens=max_tokens,
                cache_dir=cache_dir,
                enable_prefix_caching=enable_prefix_caching,
                max_model_len=max_model_len
            )
"""
mini = replace_exact(
    mini, anchor, halluguard_constructor + anchor,
    "MiniCheck HalluGuard constructor",
)

model_anchor = "        if model_id == 'Bespoke-MiniCheck-7B':\n"
inf = replace_exact(
    inf,
    model_anchor,
    "        self.model_revision = None\n        self.vllm_kwargs = {}\n\n" + model_anchor,
    "LLMCheck defaults",
)

anchor = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        else:
            raise ValueError(\"model_id must be 'Bespoke-MiniCheck-7B'\")"""
replacement = """        elif model_id == 'Granite-Guardian-3.3-8B':
            self.model_id = 'ibm-granite/granite-guardian-3.3-8b'
            self.operating_mode=\"gg_hybrid\"
        elif model_id == 'HalluGuard-Qwen3-4B':
            self.model_id = 'lrsbrgrn/HalluGuard-Qwen3-4B'
            self.operating_mode = \"halluguard_binary\"
            self.model_revision = '2bf3858f8bd0c22b0c410f6b0c855098727a2554'
            self.vllm_kwargs = {
                'enforce_eager': True,
                'disable_custom_all_reduce': True,
                'gpu_memory_utilization': 0.80,
                'max_num_seqs': 4,
            }
        else:
            raise ValueError(\"Unsupported LLMCheck model_id: %s\" % model_id)"""
inf = replace_exact(inf, anchor, replacement, "LLMCheck model dispatch")

inf = replace_exact(
    inf,
    "            model=self.model_id, ",
    "            model=self.model_id,\n            revision=self.model_revision, ",
    "vLLM revision",
)
inf = replace_exact(
    inf,
    "            enable_prefix_caching=self.enable_prefix_caching",
    "            enable_prefix_caching=self.enable_prefix_caching,\n            **self.vllm_kwargs",
    "vLLM HalluGuard stability kwargs",
)

anchor = """        self.sampling_params = SamplingParams(
            temperature=0,
            max_tokens=self.max_tokens,
            stop_token_ids=terminators,
            logprobs=5
        )"""
replacement = """        if self.operating_mode == \"halluguard_binary\":
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=[self.tokenizer.eos_token_id],
                seed=42,
            )
        else:
            self.sampling_params = SamplingParams(
                temperature=0,
                max_tokens=self.max_tokens,
                stop_token_ids=terminators,
                logprobs=5
            )
        self.last_halluguard_records = []"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard sampling parameters")

anchor = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        return text"""
replacement = """        elif self.operating_mode==\"gg_hybrid\":
            documents = [{'doc_id':'0', 'text': doc}]
            messages = [{\"role\": \"assistant\", \"content\": claim}]
            guardian_config = {\"criteria_id\": \"groundedness\"}
            text = self.tokenizer.apply_chat_template(messages, guardian_config = guardian_config, documents=documents, think=True, tokenize=False, add_generation_prompt=True)
        elif self.operating_mode == \"halluguard_binary\":
            prompt = (
                \"Document:\\n\" + doc + \"\\n\\nClaim:\\n\" + claim
                + \"\\n\\nReply exactly YES if the claim is fully supported by the document. \"
                + \"Reply exactly NO if the claim is contradicted or unsupported by the document. \"
                + \"No explanation, reasoning, punctuation, XML, or extra text.\"
            )
            text = self.tokenizer.apply_chat_template(
                [{\"role\": \"user\", \"content\": prompt}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        return text"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard chat template")

inf = replace_exact(
    inf,
    "class LLMCheck:\n",
    "class HalluGuardParseError(ValueError):\n    pass\n\n\nclass LLMCheck:\n",
    "HalluGuard parse exception",
)

parser_anchor = '    def get_support_prob_hybrid_gg(self, response, marker="score"):\n'
parser = """    def parse_halluguard_output(self, response_text):
        final_text = response_text.strip()
        if final_text == \"YES\":
            return \"GROUNDED\", 1.0, final_text
        if final_text == \"NO\":
            return \"HALLUCINATED\", 0.0, final_text
        raise HalluGuardParseError(
            \"HalluGuard binary output must be exactly trimmed uppercase YES or NO. \"
            + \"Raw output: \" + repr(response_text[:4000])
        )

    def get_support_score_halluguard(self, response):
        raw_text = response.outputs[0].text
        label, score, final_text = self.parse_halluguard_output(raw_text)
        self.last_halluguard_records.append({
            \"classification\": label,
            \"support_score\": score,
            \"raw_output\": raw_text,
        })
        return score

"""
inf = replace_exact(
    inf, parser_anchor, parser + parser_anchor,
    "HalluGuard strict parser",
)

anchor = """        if self.operating_mode==\"bespoke\":
            probs_per_chunk_sentence = [self.get_support_prob(responses[idx]) for idx in range(len(responses))]
        elif self.operating_mode==\"gg_hybrid\":
            probs_per_chunk_sentence = [self.get_support_prob_hybrid_gg(responses[idx]) for idx in range(len(responses))]"""
replacement = anchor + """
        elif self.operating_mode == \"halluguard_binary\":
            self.last_halluguard_records = []
            probs_per_chunk_sentence = [
                self.get_support_score_halluguard(response) for response in responses
            ]"""
inf = replace_exact(inf, anchor, replacement, "HalluGuard response dispatch")

ast.parse(mini, filename=str(mini_path))
ast.parse(inf, filename=str(inf_path))
mini_path.write_text(mini, encoding="utf-8")
inf_path.write_text(inf, encoding="utf-8")
print("Patched pinned MiniCheck with deterministic HalluGuard binary support.")

run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", str(SOURCE_DIR)])

import nltk
for resource in ("punkt", "punkt_tab"):
    nltk.download(resource, quiet=True)

sys.path.insert(0, str(SOURCE_DIR))
from minicheck.minicheck import MiniCheck
from minicheck.inference import HalluGuardParseError
print("MiniCheck import path:", __import__("minicheck").__file__)


In [ ]:
import json
import platform
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import balanced_accuracy_score

MODEL_NAME = "HalluGuard-Qwen3-4B"
MODEL_ID = "lrsbrgrn/HalluGuard-Qwen3-4B"
MODEL_REVISION = "2bf3858f8bd0c22b0c410f6b0c855098727a2554"
DATASET_ID = "lytang/LLM-AggreFact"
SPLIT = "test"
CHUNK_SIZE = 500
MAX_MODEL_LEN = 4096
MAX_TOKENS = 8
TENSOR_PARALLEL_SIZE = 1
ENABLE_PREFIX_CACHING = False
RANDOM_STATE = 42
CACHE_DIR = str(WORKING / "hf_cache")

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

OUTPUT_DIR = WORKING / "halluguard_minicheck_smoke"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load HalluGuard through MiniCheck


In [ ]:
scorer = MiniCheck(
    model_name="HalluGuard-Qwen3-4B",
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    max_model_len=MAX_MODEL_LEN,
    max_tokens=MAX_TOKENS,
    enable_prefix_caching=ENABLE_PREFIX_CACHING,
    cache_dir=CACHE_DIR,
)
assert scorer.model.operating_mode == "halluguard_binary"
assert scorer.model.model_id == MODEL_ID
assert scorer.model.model_revision == MODEL_REVISION
assert scorer.model.vllm_kwargs == {
    "enforce_eager": True,
    "disable_custom_all_reduce": True,
    "gpu_memory_utilization": 0.80,
    "max_num_seqs": 4,
}
print("Loaded HalluGuard through MiniCheck on", TENSOR_PARALLEL_SIZE, "GPUs.")


## 3. Strict parser contract tests

    A valid XML answer maps `GROUNDED → 1.0`; both hallucination classes map to `0.0`. Malformed, missing, ambiguous, or truncated answers raise `HalluGuardParseError` and never receive a guessed prediction.


In [ ]:
for raw, expected_label, expected_score in [(" YES ", "GROUNDED", 1.0), ("NO", "HALLUCINATED", 0.0)]:
    label, score, _ = scorer.model.parse_halluguard_output(raw)
    assert (label, score) == (expected_label, expected_score)
try:
    scorer.model.parse_halluguard_output("<answer>GROUNDED</answer>")
    raise AssertionError("Malformed binary output was accepted")
except HalluGuardParseError:
    pass
print("Strict YES/NO parser contract passed.")


## 4. Handcrafted scoring and chunking checks


In [ ]:
long_doc = " ".join([
    f"Sentence {i} states that the observatory recorded a blue comet on Tuesday and no meteor on Wednesday."
    for i in range(55)
])
handcrafted = pd.DataFrame([
    {"case": "grounded", "doc": "The lab opens at 08:00 and closes at 17:00.", "claim": "The lab opens at 08:00.", "expected": 1},
    {"case": "intrinsic", "doc": "The lab opens at 08:00 and closes at 17:00.", "claim": "The lab opens at 10:00.", "expected": 0},
    {"case": "extrinsic", "doc": "The lab opens at 08:00 and closes at 17:00.", "claim": "The lab director was born in Paris.", "expected": 0},
    {"case": "multi_chunk", "doc": long_doc, "claim": "The observatory recorded a blue comet on Tuesday.", "expected": 1},
])

preds, probs, chunks, chunk_probs = scorer.score(
    docs=handcrafted["doc"].tolist(),
    claims=handcrafted["claim"].tolist(),
    chunk_size=CHUNK_SIZE,
)
handcrafted["prediction"] = preds
handcrafted["support_score"] = probs
handcrafted["n_chunks"] = [len(x) for x in chunks]
handcrafted["chunk_sentence_shape"] = [str(np.asarray(x).shape) for x in chunk_probs]
assert all(p in (0, 1) for p in preds)
assert all(float(p) in (0.0, 1.0) for p in probs)
assert handcrafted.loc[handcrafted["case"] == "multi_chunk", "n_chunks"].iloc[0] > 1
assert scorer.model.last_halluguard_records
handcrafted.to_csv(OUTPUT_DIR / "handcrafted_results.csv", index=False)
display(handcrafted.drop(columns=["doc"]))


## 5. One test example from every LLM-AggreFact dataset

    Sampling is deterministic. Each row is scored through `MiniCheck.score()` with `chunk_size=500`; exceptions are recorded without fabricated predictions.


In [ ]:
raw = load_dataset(DATASET_ID, split=SPLIT, token=hf_token)
full_df = raw.to_pandas()
full_df["source_index"] = np.arange(len(full_df), dtype=np.int64)
dataset_names = sorted(full_df["dataset"].unique().tolist())
assert len(dataset_names) == 11, f"Expected 11 datasets, found {len(dataset_names)}: {dataset_names}"
smoke_df = pd.concat([
    group.sample(n=1, random_state=RANDOM_STATE)
    for _, group in full_df.groupby("dataset", sort=True)
], ignore_index=True)

records = []
for row in smoke_df.itertuples(index=False):
    started = time.perf_counter()
    base = {
        "dataset": row.dataset,
        "source_index": int(row.source_index),
        "label": int(row.label),
        "prediction": None,
        "support_score": None,
        "n_chunks": None,
        "status": "INFERENCE_FAILURE",
        "error_type": None,
        "error_message": None,
    }
    try:
        preds, probs, chunks, matrices = scorer.score(
            docs=[str(row.doc)], claims=[str(row.claim)], chunk_size=CHUNK_SIZE
        )
        base.update({
            "prediction": int(preds[0]),
            "support_score": float(probs[0]),
            "n_chunks": len(chunks[0]),
            "chunk_sentence_shape": str(np.asarray(matrices[0]).shape),
            "status": "OK",
        })
    except HalluGuardParseError as exc:
        base.update(status="PARSE_FAILURE", error_type=type(exc).__name__, error_message=str(exc)[:4000])
    except Exception as exc:
        base.update(error_type=type(exc).__name__, error_message=str(exc)[:4000])
    base["elapsed_seconds"] = time.perf_counter() - started
    records.append(base)
    pd.DataFrame(records).to_csv(OUTPUT_DIR / "dataset_smoke_results.csv", index=False)
    print(row.dataset, base["status"], base["prediction"], base["support_score"])

smoke_results = pd.DataFrame(records)
assert len(smoke_results) == 11 and smoke_results["dataset"].nunique() == 11
assert (smoke_results["status"] == "OK").all(), smoke_results.loc[smoke_results["status"] != "OK"]
display(smoke_results)


## 6. Save diagnostics and manifest


In [ ]:
gpu_after = subprocess.run(["nvidia-smi"], text=True, capture_output=True, check=True).stdout
(OUTPUT_DIR / "nvidia_smi_before.txt").write_text(gpu_before, encoding="utf-8")
(OUTPUT_DIR / "nvidia_smi_after.txt").write_text(gpu_after, encoding="utf-8")

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "HalluGuard MiniCheck integration smoke test",
    "minicheck_commit": MINICHECK_COMMIT,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset": DATASET_ID,
    "split": SPLIT,
    "chunk_size": CHUNK_SIZE,
    "max_model_len": MAX_MODEL_LEN,
    "max_tokens": MAX_TOKENS,
    "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
    "enable_prefix_caching": ENABLE_PREFIX_CACHING,
    "vllm_engine": {
        "version": "0.10.2",
        "engine_mode": "V0",
        "enforce_eager": True,
        "disable_custom_all_reduce": True,
        "gpu_memory_utilization": 0.80,
        "max_num_seqs": 4,
        "reason": "Single-T4 eager-mode stability after observed tensor-parallel engine failures",
    },
    "random_state": RANDOM_STATE,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "handcrafted_rows": len(handcrafted),
    "benchmark_smoke_rows": len(smoke_results),
    "parse_failures": int((smoke_results["status"] == "PARSE_FAILURE").sum()),
    "inference_failures": int((smoke_results["status"] == "INFERENCE_FAILURE").sum()),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("Smoke test actual counts:", {"total": len(smoke_results), "successful": int((smoke_results["status"] == "OK").sum()), "parse_failures": manifest["parse_failures"], "inference_failures": manifest["inference_failures"]})
print("Outputs:", OUTPUT_DIR)
